In [ ]:
import pandas as pd

In [ ]:
inspections = pd.read_csv("Inspections.csv")
bridges = pd.read_csv("MA_bridges_2000_2025.csv")

In [ ]:
"""
Download NBI (National Bridge Inventory) delimited ASCII files for all available
years, filter for Massachusetts (state code 25), and combine into a single CSV.

Source: https://www.fhwa.dot.gov/bridge/nbi/ascii.cfm

File URL pattern (verified):
    https://www.fhwa.dot.gov/bridge/nbi/{year}/delimited/MA{yy}.txt

Note: file naming/format has changed slightly across years (older years used a
slightly different directory naming and some pre-2000 years may not be available
in this exact format). The script below handles the common 1992-2025 pattern and
will skip/report any year that fails to download, so you can adjust as needed.

The downloaded files already have header rows with column names that match the
NBI Coding Guide item numbers (e.g. STRUCTURE_NUMBER_008 = Item 8). This script
keeps those original column names since they already encode the NBI item number,
and optionally adds a more descriptive label using a mapping dict for the most
commonly used columns.
"""

import requests
import pandas as pd
import io
import time

# -----------------------------------------------------------------------
# Config
# -----------------------------------------------------------------------
START_YEAR = 1992
END_YEAR = 2025          # change as needed; will skip years that 404
STATE_ABBR = "MA"
STATE_CODE = "25"        # NBI numeric state code for Massachusetts
OUTPUT_FILE = "massachusetts_bridges_all_years.csv"

BASE_URL = "https://www.fhwa.dot.gov/bridge/nbi/{year}/delimited/{state}{yy}.txt"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (research data download script)"
}

# -----------------------------------------------------------------------
# Optional: human-readable labels for common NBI item columns.
# The raw column headers already contain the item number (e.g. "_008"),
# so this is just an additional descriptive label layer you can merge in
# if you want fuller names. Extend this as needed from the NBI Coding Guide
# / nationalbridges.com data dictionary.
# -----------------------------------------------------------------------
NBI_COLUMN_LABELS = {
    "STATE_CODE_001": "State Code",
    "STRUCTURE_NUMBER_008": "Structure Number",
    "RECORD_TYPE_005A": "Record Type",
    "ROUTE_PREFIX_005B": "Route Signing Prefix",
    "SERVICE_LEVEL_005C": "Designated Level of Service",
    "ROUTE_NUMBER_005D": "Route Number",
    "DIRECTION_005E": "Directional Suffix",
    "HIGHWAY_DISTRICT_002": "Highway Agency District",
    "COUNTY_CODE_003": "County Code",
    "PLACE_CODE_004": "Place Code",
    "FEATURES_DESC_006A": "Features Intersected",
    "CRITICAL_FACILITY_006B": "Critical Facility Indicator",
    "FACILITY_CARRIED_007": "Facility Carried by Structure",
    "LOCATION_009": "Location",
    "MIN_VERT_CLR_010": "Inventory Route Min Vertical Clearance",
    "KILOPOINT_011": "Kilometerpoint",
    "BASE_HWY_NETWORK_012": "Base Highway Network",
    "LRS_INV_ROUTE_013A": "LRS Inventory Route",
    "SUBROUTE_NO_013B": "Subroute Number",
    "LAT_016": "Latitude",
    "LONG_017": "Longitude",
    "DETOUR_KILOS_019": "Bypass/Detour Length",
    "TOLL_020": "Toll",
    "MAINTENANCE_021": "Maintenance Responsibility",
    "OWNER_022": "Owner",
    "FUNCTIONAL_CLASS_026": "Functional Classification",
    "YEAR_BUILT_027": "Year Built",
    "TRAFFIC_LANES_ON_028A": "Lanes On Structure",
    "TRAFFIC_LANES_UND_028B": "Lanes Under Structure",
    "ADT_029": "Average Daily Traffic",
    "YEAR_ADT_030": "Year of ADT",
    "DESIGN_LOAD_031": "Design Load",
    "APPR_WIDTH_MT_032": "Approach Roadway Width (m)",
    "MEDIAN_CODE_033": "Bridge Median",
    "DEGREES_SKEW_034": "Skew",
    "STRUCTURE_FLARED_035": "Structure Flared",
    "RAILINGS_036A": "Bridge Railings",
    "TRANSITIONS_036B": "Transitions",
    "APPR_RAIL_036C": "Approach Guardrail",
    "APPR_RAIL_END_036D": "Approach Guardrail Ends",
    "HISTORY_037": "Historical Significance",
    "NAVIGATION_038": "Navigation Control",
    "NAV_VERT_CLR_MT_039": "Navigation Vertical Clearance (m)",
    "NAV_HORR_CLR_MT_040": "Navigation Horizontal Clearance (m)",
    "OPEN_CLOSED_POSTED_041": "Structure Open/Posted/Closed",
    "SERVICE_ON_042A": "Type of Service On Bridge",
    "SERVICE_UND_042B": "Type of Service Under Bridge",
    "STRUCTURE_KIND_043A": "Structure Kind, Main",
    "STRUCTURE_TYPE_043B": "Structure Type, Main",
    "APPR_KIND_044A": "Structure Kind, Approach",
    "APPR_TYPE_044B": "Structure Type, Approach",
    "MAIN_UNIT_SPANS_045": "Number of Spans in Main Unit",
    "APPR_SPANS_046": "Number of Approach Spans",
    "HORR_CLR_MT_047": "Inventory Route Total Horizontal Clearance (m)",
    "MAX_SPAN_LEN_MT_048": "Length of Maximum Span (m)",
    "STRUCTURE_LEN_MT_049": "Structure Length (m)",
    "LEFT_CURB_MT_050A": "Left Curb/Sidewalk Width (m)",
    "RIGHT_CURB_MT_050B": "Right Curb/Sidewalk Width (m)",
    "ROADWAY_WIDTH_MT_051": "Bridge Roadway Width Curb-to-Curb (m)",
    "DECK_WIDTH_MT_052": "Deck Width Out-to-Out (m)",
    "VERT_CLR_OVER_MT_053": "Min Vertical Clearance Over Bridge Roadway (m)",
    "VERT_CLR_UND_REF_054A": "Min Vertical Underclearance Reference Feature",
    "VERT_CLR_UND_054B": "Min Vertical Underclearance (m)",
    "LAT_UND_REF_055A": "Min Lateral Underclearance Reference Feature",
    "LAT_UND_MT_055B": "Min Lateral Underclearance Right (m)",
    "LEFT_LAT_UND_MT_056": "Min Lateral Underclearance Left (m)",
    "DECK_COND_058": "Deck Condition Rating",
    "SUPERSTRUCTURE_COND_059": "Superstructure Condition Rating",
    "SUBSTRUCTURE_COND_060": "Substructure Condition Rating",
    "CHANNEL_COND_061": "Channel and Channel Protection",
    "CULVERT_COND_062": "Culvert Condition Rating",
    "OPR_RATING_METH_063": "Method Used to Determine Operating Rating",
    "OPERATING_RATING_064": "Operating Rating",
    "INV_RATING_METH_065": "Method Used to Determine Inventory Rating",
    "INVENTORY_RATING_066": "Inventory Rating",
    "STRUCTURAL_EVAL_067": "Structural Evaluation Appraisal",
    "DECK_GEOMETRY_EVAL_068": "Deck Geometry Appraisal",
    "UNDCLRENCE_EVAL_069": "Underclearances Appraisal",
    "POSTING_EVAL_070": "Bridge Posting",
    "WATERWAY_EVAL_071": "Waterway Adequacy Appraisal",
    "APPR_ROAD_EVAL_072": "Approach Roadway Alignment Appraisal",
    "WORK_PROPOSED_075A": "Type of Work Proposed",
    "WORK_DONE_BY_075B": "Work Done By",
    "IMP_LEN_MT_076": "Length of Structure Improvement (m)",
    "DATE_OF_INSPECT_090": "Inspection Date",
    "INSPECT_FREQ_MONTHS_091": "Designated Inspection Frequency (months)",
    "FRACTURE_092A": "Fracture Critical Details Inspection",
    "UNDWATER_LOOK_SEE_092B": "Underwater Inspection",
    "SPEC_INSPECT_092C": "Other Special Inspection",
    "FRACTURE_LAST_DATE_093A": "Fracture Critical Last Inspection Date",
    "UNDWATER_LAST_DATE_093B": "Underwater Last Inspection Date",
    "SPEC_LAST_DATE_093C": "Special Inspection Last Date",
    "BRIDGE_IMP_COST_094": "Bridge Improvement Cost ($1000s)",
    "ROADWAY_IMP_COST_095": "Roadway Improvement Cost ($1000s)",
    "TOTAL_IMP_COST_096": "Total Project Cost ($1000s)",
    "YEAR_OF_IMP_097": "Year of Improvement Cost Estimate",
    "OTHER_STATE_CODE_098A": "Border Bridge - Neighboring State Code",
    "OTHER_STATE_PCNT_098B": "Border Bridge - Percent Responsibility",
    "OTHR_STATE_STRUC_NO_099": "Border Bridge Structure Number",
    "STRAHNET_HIGHWAY_100": "STRAHNET Highway Designation",
    "PARALLEL_STRUCTURE_101": "Parallel Structure Designation",
    "TRAFFIC_DIRECTION_102": "Direction of Traffic",
    "TEMP_STRUCTURE_103": "Temporary Structure Designation",
    "HIGHWAY_SYSTEM_104": "Highway System of Inventory Route",
    "FEDERAL_LANDS_105": "Federal Lands Highway",
    "YEAR_RECONSTRUCTED_106": "Year Reconstructed",
    "DECK_STRUCTURE_TYPE_107": "Deck Structure Type",
    "SURFACE_TYPE_108A": "Type of Wearing Surface",
    "MEMBRANE_TYPE_108B": "Type of Membrane",
    "DECK_PROTECTION_108C": "Deck Protection",
    "PERCENT_ADT_TRUCK_109": "Average Daily Truck Traffic (%)",
    "NATIONAL_NETWORK_110": "Designated National Network",
    "PIER_PROTECTION_111": "Pier or Abutment Protection",
    "BRIDGE_LEN_IND_112": "NBIS Bridge Length Indicator",
    "SCOUR_CRITICAL_113": "Scour Critical Bridges",
    "FUTURE_ADT_114": "Future Average Daily Traffic",
    "YEAR_OF_FUTURE_ADT_115": "Year of Future ADT",
    "MIN_NAV_CLR_MT_116": "Minimum Navigation Vertical Clearance (m)",
    "FED_AGENCY": "Federal Agency",
    "SUBMITTED_BY": "Submitted By",
    "BRIDGE_CONDITION": "Overall Bridge Condition",
    "LOWEST_RATING": "Lowest Condition Rating",
    "DECK_AREA": "Deck Area",
}


def download_year(year: int) -> pd.DataFrame | None:
    """Download and parse the MA delimited file for a given year."""
    yy = f"{year % 100:02d}"
    url = BASE_URL.format(year=year, state=STATE_ABBR, yy=yy)

    try:
        resp = requests.get(url, headers=HEADERS, timeout=60)
        resp.raise_for_status()
    except requests.RequestException as e:
        print(f"  [{year}] Failed to download: {e}")
        return None

    text = resp.text
    if not text.strip():
        print(f"  [{year}] Empty response, skipping.")
        return None

    try:
        # Files use a single quote as text qualifier per FHWA's note
        df = pd.read_csv(io.StringIO(text), quotechar="'", low_memory=False)
    except Exception as e:
        print(f"  [{year}] Failed to parse CSV: {e}")
        return None

    df["SURVEY_YEAR"] = year  # track which year's submission this row came from

    # Filter for MA just in case "all states" or mis-tagged rows sneak in
    state_col = None
    for candidate in ["STATE_CODE_001", "STATE_CODE"]:
        if candidate in df.columns:
            state_col = candidate
            break

    if state_col:
        df = df[df[state_col].astype(str).str.zfill(2) == STATE_CODE]

    print(f"  [{year}] Downloaded {len(df)} MA rows.")
    return df


def main():
    all_dfs = []

    for year in range(START_YEAR, END_YEAR + 1):
        print(f"Processing {year}...")
        df = download_year(year)
        if df is not None and not df.empty:
            all_dfs.append(df)
        time.sleep(0.5)  # be polite to the server

    if not all_dfs:
        print("No data downloaded. Exiting.")
        return

    combined = pd.concat(all_dfs, ignore_index=True, sort=False)

    # Apply descriptive labels as a second header row / or rename columns directly.
    # Option A (uncomment to RENAME columns to descriptive labels, keeping item numbers
    # would be lost from the column name itself but is preserved in NBI_COLUMN_LABELS keys):
    #
    # combined = combined.rename(columns=NBI_COLUMN_LABELS)

    # Option B (default): keep original NBI column names (which already include
    # the item number, e.g. STRUCTURE_NUMBER_008) and write out a separate
    # label mapping file for reference.
    combined.to_csv(OUTPUT_FILE, index=False)
    print(f"\nSaved combined data to {OUTPUT_FILE} ({len(combined)} rows, "
          f"{len(combined.columns)} columns).")

    # Write the column label reference as its own CSV
    label_df = pd.DataFrame(
        [{"column_name": col, "description": NBI_COLUMN_LABELS.get(col, "")}
         for col in combined.columns]
    )
    label_df.to_csv("nbi_column_labels.csv", index=False)
    print("Saved column label reference to nbi_column_labels.csv")


if __name__ == "__main__":
    main()

In [ ]:
ma_bridges = pd.read_csv("massachusetts_bridges_all_years.csv")
print(ma_bridges.head(20))

In [ ]:
print(inspections.head(10))

In [ ]:
# ---------------------------------------------------------------
# Load data
# ---------------------------------------------------------------
bridges = pd.read_csv("massachusetts_bridges_all_years.csv", low_memory=False)
maint = pd.read_csv("maintenance.csv")  # adjust filename/path as needed

# ---------------------------------------------------------------
# Parse maintenance dates
# ---------------------------------------------------------------
maint["AD_DATE_ACTUAL"] = pd.to_datetime(maint["AD_DATE_ACTUAL"], errors="coerce")
maint["COMPLETE_DATE"] = pd.to_datetime(maint["COMPLETE_DATE"], errors="coerce")

today = pd.Timestamp.today()

# ---------------------------------------------------------------
# Identify ongoing vs completed maintenance
# A project is "ongoing" if it has no completion date, or the
# completion date is in the future relative to today (i.e. the
# work hasn't actually finished yet, even if a target date is listed).
# ---------------------------------------------------------------
maint["is_ongoing"] = maint["COMPLETE_DATE"].isna() | (maint["COMPLETE_DATE"] > today)

completed = maint[~maint["is_ongoing"]].copy()
completed["completion_year"] = completed["COMPLETE_DATE"].dt.year

# A bridge can have multiple completed maintenance projects across years.
# Keep one row per (bridge, completion_year) since that's all we need.
completed_unique = (
    completed[["Item8", "completion_year"]]
    .drop_duplicates()
    .rename(columns={"Item8": "STRUCTURE_NUMBER_008"})
    .sort_values("completion_year")
)

# ---------------------------------------------------------------
# merge_asof: for each bridge-year row, find the most recent
# completed maintenance year that is STRICTLY BEFORE that survey year.
# (allow_exact_matches=False -> matches your "after 2020" rule:
#  a 2020 maintenance only shows up as "had maintenance" starting 2021)
# ---------------------------------------------------------------
bridges_sorted = bridges.sort_values("SURVEY_YEAR")
bridges_sorted["SURVEY_YEAR"] = bridges_sorted["SURVEY_YEAR"].astype("int64")
completed_unique["completion_year"] = completed_unique["completion_year"].astype("int64")

merged = pd.merge_asof(
    bridges_sorted,
    completed_unique,
    left_on="SURVEY_YEAR",
    right_on="completion_year",
    by="STRUCTURE_NUMBER_008",
    direction="backward",
    allow_exact_matches=False,
)

merged["had_maintenance"] = merged["completion_year"].notna().astype(int)
merged["time_since_last_maintenance"] = merged["SURVEY_YEAR"] - merged["completion_year"]

merged = merged.drop(columns=["completion_year"])

# ---------------------------------------------------------------
# Optional: flag bridges currently undergoing maintenance in a given
# survey year (started, but not yet completed as of that year).
# Build (bridge, year) pairs covering every year a project was "in progress".
# ---------------------------------------------------------------
ongoing = maint[maint["is_ongoing"]].copy()
ongoing["start_year"] = ongoing["AD_DATE_ACTUAL"].dt.year
# If no completion date, treat as still in progress through the most recent
# survey year in the dataset. If a future completion date exists, use that
# as the upper bound (capped at the max survey year, since we don't have
# observations past that anyway).
max_survey_year = bridges["SURVEY_YEAR"].max()
ongoing["end_year"] = ongoing["COMPLETE_DATE"].dt.year.fillna(max_survey_year)
ongoing["end_year"] = ongoing["end_year"].clip(upper=max_survey_year)

ongoing_pairs = []
for _, row in ongoing.iterrows():
    if pd.isna(row["start_year"]):
        continue
    for yr in range(int(row["start_year"]), int(row["end_year"]) + 1):
        ongoing_pairs.append((row["Item8"], yr))

ongoing_df = (
    pd.DataFrame(ongoing_pairs, columns=["STRUCTURE_NUMBER_008", "SURVEY_YEAR"])
    .drop_duplicates()
)
ongoing_df["maintenance_in_progress"] = 1

merged = merged.merge(
    ongoing_df, on=["STRUCTURE_NUMBER_008", "SURVEY_YEAR"], how="left"
)
merged["maintenance_in_progress"] = merged["maintenance_in_progress"].fillna(0).astype(int)

merged.to_csv("massachusetts_bridges_with_maintenance.csv", index=False)


In [ ]:

print(merged[["STRUCTURE_NUMBER_008", "SURVEY_YEAR", "had_maintenance",
               "time_since_last_maintenance", "maintenance_in_progress"]].tail(20))

In [ ]:
merged.to_csv("merged.csv")

In [ ]:
"""
Pull Daymet climate data for MA bridge locations and compute derived
annual climate indicators (freeze-thaw cycles, freezing index,
precipitation totals, snowpack, seasonal temperatures) to merge with
the NBI bridge dataset for survival modeling.

Requires: pip install pydaymet pandas numpy
"""

import pandas as pd
import numpy as np
import pydaymet as daymet
import time

# -----------------------------------------------------------------------
# Config
# -----------------------------------------------------------------------
BRIDGE_FILE = "massachusetts_bridges_with_maintenance.csv"
OUTPUT_FILE = "massachusetts_bridges_with_climate.csv"
START_YEAR = 1992
END_YEAR = 2025  # Daymet has a lag of ~1 year for final data; adjust if 2025 fails
GRID_PRECISION = 2  # rounding decimals (~1.1 km at MA's latitude)

DAYMET_VARS = ["tmax", "tmin", "prcp", "swe"]

# -----------------------------------------------------------------------
# 1. Load bridges and convert NBI lat/long (DDMMSSss / DDDMMSSss format)
#    to decimal degrees
# -----------------------------------------------------------------------
bridges = pd.read_csv(BRIDGE_FILE, low_memory=False)


def nbi_lat_to_decimal(val):
    """Convert 8-digit NBI latitude (DDMMSSss) to decimal degrees."""
    if pd.isna(val):
        return np.nan
    s = f"{int(val):08d}"
    deg = int(s[0:2])
    minute = int(s[2:4])
    sec = int(s[4:6]) + int(s[6:8]) / 100
    return deg + minute / 60 + sec / 3600


def nbi_lon_to_decimal(val):
    """Convert 9-digit NBI longitude (DDDMMSSss) to decimal degrees (West -> negative)."""
    if pd.isna(val):
        return np.nan
    s = f"{int(val):09d}"
    deg = int(s[0:3])
    minute = int(s[3:5])
    sec = int(s[5:7]) + int(s[7:9]) / 100
    return -(deg + minute / 60 + sec / 3600)


# One row per bridge with its coordinates (lat/long shouldn't change across years,
# but use the first non-null record just in case)
coords = (
    bridges.dropna(subset=["LAT_016", "LONG_017"])
    .groupby("STRUCTURE_NUMBER_008")[["LAT_016", "LONG_017"]]
    .first()
    .reset_index()
)

coords["lat"] = coords["LAT_016"].apply(nbi_lat_to_decimal)
coords["lon"] = coords["LONG_017"].apply(nbi_lon_to_decimal)

# Drop obviously bad coordinates (0,0 or outside MA's rough bounding box)
coords = coords[
    (coords["lat"].between(41, 43)) & (coords["lon"].between(-74, -69))
].copy()

# Snap to Daymet grid cell to reduce duplicate API calls
coords["grid_lat"] = coords["lat"].round(GRID_PRECISION)
coords["grid_lon"] = coords["lon"].round(GRID_PRECISION)

unique_cells = coords[["grid_lat", "grid_lon"]].drop_duplicates().reset_index(drop=True)
print(f"{len(coords)} bridges -> {len(unique_cells)} unique Daymet grid cells")

# -----------------------------------------------------------------------
# 2. Pull daily Daymet data for each unique grid cell
# -----------------------------------------------------------------------
dates = (f"{START_YEAR}-01-01", f"{END_YEAR}-12-31")

daily_frames = []

for i, row in unique_cells.iterrows():
    lat, lon = row["grid_lat"], row["grid_lon"]
    try:
        df = daymet.get_bycoords(
            (lon, lat),
            dates=dates,
            variables=DAYMET_VARS,
            crs=4326,
        )
    except Exception as e:
        print(f"  [{i}] Failed for ({lat}, {lon}): {e}")
        continue

    df = df.reset_index().rename(columns={"index": "date", "time": "date"})
    df["grid_lat"] = lat
    df["grid_lon"] = lon
    daily_frames.append(df)

    if i % 50 == 0:
        print(f"  Retrieved {i+1}/{len(unique_cells)} grid cells")

    time.sleep(0.1)  # be polite to the API

daily = pd.concat(daily_frames, ignore_index=True)
daily["date"] = pd.to_datetime(daily["date"])
daily["year"] = daily["date"].dt.year
daily["month"] = daily["date"].dt.month

# -----------------------------------------------------------------------
# 3. Compute derived annual climate indicators per grid cell
# -----------------------------------------------------------------------
# Freeze-thaw cycle: a day where tmin < 0C and tmax > 0C
daily["freeze_thaw_day"] = (daily["tmin (deg c)"] < 0) & (daily["tmax (deg c)"] > 0)

# Freezing index: cumulative degree-days below 0C (simple proxy using tmin)
daily["freeze_degree_day"] = np.where(
    daily["tmin (deg c)"] < 0, -daily["tmin (deg c)"], 0
)

# Snow day proxy: precipitation on a day where tmax < 0C (likely fell as snow)
daily["snow_day"] = (daily["prcp (mm/day)"] > 0) & (daily["tmax (deg c)"] < 0)

# Winter (DJF) / Summer (JJA) flags for seasonal temperature means
daily["season"] = daily["month"].map(
    {12: "DJF", 1: "DJF", 2: "DJF", 6: "JJA", 7: "JJA", 8: "JJA"}
)

annual = (
    daily.groupby(["grid_lat", "grid_lon", "year"])
    .agg(
        freeze_thaw_cycles=("freeze_thaw_day", "sum"),
        freezing_index=("freeze_degree_day", "sum"),
        annual_precip_mm=("prcp (mm/day)", "sum"),
        snow_days=("snow_day", "sum"),
        max_swe=("swe (kg/m^2)", "max"),
    )
    .reset_index()
)

seasonal = (
    daily.dropna(subset=["season"])
    .groupby(["grid_lat", "grid_lon", "year", "season"])
    .agg(
        mean_tmax=("tmax (deg c)", "mean"),
        mean_tmin=("tmin (deg c)", "mean"),
    )
    .reset_index()
)

seasonal_wide = seasonal.pivot_table(
    index=["grid_lat", "grid_lon", "year"],
    columns="season",
    values=["mean_tmax", "mean_tmin"],
)
seasonal_wide.columns = [f"{a}_{b}" for a, b in seasonal_wide.columns]
seasonal_wide = seasonal_wide.reset_index()

annual = annual.merge(seasonal_wide, on=["grid_lat", "grid_lon", "year"], how="left")

# -----------------------------------------------------------------------
# 4. Merge climate data back to bridges via grid cell, then onto the
#    full bridge-year dataset via STRUCTURE_NUMBER_008 + SURVEY_YEAR
# -----------------------------------------------------------------------
bridge_cells = coords[["STRUCTURE_NUMBER_008", "grid_lat", "grid_lon"]]

bridge_climate = bridge_cells.merge(annual, on=["grid_lat", "grid_lon"], how="left")
bridge_climate = bridge_climate.rename(columns={"year": "SURVEY_YEAR"})

merged = bridges.merge(
    bridge_climate.drop(columns=["grid_lat", "grid_lon"]),
    on=["STRUCTURE_NUMBER_008", "SURVEY_YEAR"], 
    how="left",
)

merged.to_csv(OUTPUT_FILE, index=False)
print(f"Saved {OUTPUT_FILE} with {len(merged)} rows and {len(merged.columns)} columns.")

In [ ]:
# Auto-detect and standardize column names (handles "tmin" or "tmin (deg c)" etc.)
rename_map = {}
for col in daily.columns:
    base = col.split(" ")[0].lower()
    if base in ["tmax", "tmin", "prcp", "swe", "srad", "vp", "dayl"]:
        rename_map[col] = base

daily = daily.rename(columns=rename_map)
print(daily.columns.tolist())  # confirm: should now show tmax, tmin, prcp, swe

In [ ]:
print(daily.columns.tolist())

In [ ]:
# -----------------------------------------------------------------------
# 3. Compute derived annual climate indicators per grid cell
# -----------------------------------------------------------------------
daily["freeze_thaw_day"] = (daily["tmin"] < 0) & (daily["tmax"] > 0)
daily["freeze_degree_day"] = np.where(daily["tmin"] < 0, -daily["tmin"], 0)
daily["snow_day"] = (daily["prcp"] > 0) & (daily["tmax"] < 0)

daily["season"] = daily["month"].map(
    {12: "DJF", 1: "DJF", 2: "DJF", 6: "JJA", 7: "JJA", 8: "JJA"}
)

annual = (
    daily.groupby(["grid_lat", "grid_lon", "year"])
    .agg(
        freeze_thaw_cycles=("freeze_thaw_day", "sum"),
        freezing_index=("freeze_degree_day", "sum"),
        annual_precip_mm=("prcp", "sum"),
        snow_days=("snow_day", "sum"),
        max_swe=("swe", "max"),
    )
    .reset_index()
)

seasonal = (
    daily.dropna(subset=["season"])
    .groupby(["grid_lat", "grid_lon", "year", "season"])
    .agg(mean_tmax=("tmax", "mean"), mean_tmin=("tmin", "mean"))
    .reset_index()
)

seasonal_wide = seasonal.pivot_table(
    index=["grid_lat", "grid_lon", "year"],
    columns="season",
    values=["mean_tmax", "mean_tmin"],
)
seasonal_wide.columns = [f"{a}_{b}" for a, b in seasonal_wide.columns]
seasonal_wide = seasonal_wide.reset_index()

annual = annual.merge(seasonal_wide, on=["grid_lat", "grid_lon", "year"], how="left")

# -----------------------------------------------------------------------
# 4. Merge back into bridges
# -----------------------------------------------------------------------
bridge_cells = coords[["STRUCTURE_NUMBER_008", "grid_lat", "grid_lon"]]
bridge_climate = bridge_cells.merge(annual, on=["grid_lat", "grid_lon"], how="left")
bridge_climate = bridge_climate.rename(columns={"year": "SURVEY_YEAR"})

merged = bridges.merge(
    bridge_climate.drop(columns=["grid_lat", "grid_lon"]),
    on=["STRUCTURE_NUMBER_008", "SURVEY_YEAR"],
    how="left",
)

merged.to_csv("massachusetts_bridges_with_climate.csv", index=False)
print(f"Saved: {len(merged)} rows, {len(merged.columns)} columns.")

In [ ]:
# Check how many rows have missing climate data (e.g., bridges that fell outside MA bbox or had failed grid cells)
print(merged[['freeze_thaw_cycles','annual_precip_mm']].isna().sum())

# Sanity check the value ranges - freeze_thaw_cycles should be roughly 20-70 for MA
print(merged['freeze_thaw_cycles'].describe())
print(merged['annual_precip_mm'].describe())

In [ ]:
missing = merged[merged['freeze_thaw_cycles'].isna()]
print(missing['STRUCTURE_NUMBER_008'].nunique(), "unique bridges affected")
print(missing['SURVEY_YEAR'].value_counts().sort_index())
print(missing[['STRUCTURE_NUMBER_008','LAT_016','LONG_017']].drop_duplicates().head(10))

In [ ]:
df = pd.read_csv("massachusetts_bridges_with_climate.csv")

In [ ]:
print(df.columns.tolist())

In [ ]:
df.head(10).to_csv("test_df.csv")

In [ ]:
missing = df[df['freeze_thaw_cycles'].isna()]
print(missing['STRUCTURE_NUMBER_008'].nunique(), "unique bridges affected")
print(missing['SURVEY_YEAR'].value_counts().sort_index())
print(missing[['STRUCTURE_NUMBER_008','LAT_016','LONG_017']].drop_duplicates().head(10))

In [ ]:
"""
Clean the combined MA bridge dataset (NBI + maintenance + Daymet climate)
for use in a Random Survival Forest model.

Steps:
1. Drop columns that are administrative/free-text/redundant/leakage-prone
   for a deterioration-prediction model.
2. Convert NBI sentinel "missing" codes (99, 99.99, 999, 'N', etc.) to NaN
   per the NBI data dictionary (nationalbridges.com/nbiDesc.html), while
   preserving legitimate non-missing meanings (e.g. "unrestricted",
   "never reconstructed") as separate flag columns.
3. Fix dtypes (numeric vs categorical vs string) for modeling.
4. Handle structural NaNs that are legitimately "not applicable" (e.g.
   CULVERT_COND_062 = 'N' for non-culvert bridges) vs true missing data.
5. Engineer bridge_age (key survival covariate).
6. Final missingness report so you can decide on imputation/drop thresholds
   before modeling.

All raw NBI numeric/code columns are kept as-is (not converted to text
labels) since RSF/tree-based models work directly with numeric codes.
"""

import pandas as pd
import numpy as np

INPUT_FILE = "massachusetts_bridges_with_climate.csv"   # adjust path as needed
OUTPUT_FILE = "massachusetts_bridges_rsf_ready.csv"

df = pd.read_csv(INPUT_FILE, low_memory=False)

cols = [
    'DECK_COND_058',
    'SUPERSTRUCTURE_COND_059',
    'SUBSTRUCTURE_COND_060'
]

# Replace N with 10
df[cols] = df[cols].replace('N', 10)

# Convert to numeric
df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')

df['event'] = (
    (df['DECK_COND_058'] <= 4) |
    (df['SUPERSTRUCTURE_COND_059'] <= 4) |
    (df['SUBSTRUCTURE_COND_060'] <= 4)
).astype(int)

# Drop the unnamed index column pandas sometimes writes out
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

print(f"Starting shape: {df.shape}")

# -----------------------------------------------------------------------
# 1. Drop columns not useful for RSF modeling
# -----------------------------------------------------------------------
# Reasoning per group:
# - Pure administrative/bookkeeping fields (no predictive content)
# - Free-text narrative fields (not usable without NLP, out of scope here)
# - Near-duplicate / redundant identifiers
# - Direct leakage risk: fields that are appraisal *outcomes* derived from
#   or highly correlated with condition (already discussed for the Cox
#   model - same logic applies here)
# - Sparse/legacy fields rarely populated post-1995

drop_cols = [
    # --- administrative / bookkeeping, no engineering meaning ---
    "DATE_LAST_UPDATE", "TYPE_LAST_UPDATE", "DEDUCT_CODE", "REMARKS",
    "PROGRAM_CODE", "PROJ_NO", "PROJ_SUFFIX", "NBI_TYPE_OF_IMP",
    "DTL_TYPE_OF_IMP", "SPECIAL_CODE", "STEP_CODE", "FED_AGENCY",
    "SUBMITTED_BY", "CAT10", "CAT23", "CAT29",

    # --- free text / narrative, not usable directly ---
    "FEATURES_DESC_006A", "FACILITY_CARRIED_007", "LOCATION_009",

    # --- near-duplicate identifiers / redundant geography ---
    "STATE_CODE_001",      # constant - all MA (25)
    "ROUTE_NUMBER_005D", "ROUTE_PREFIX_005B", "SERVICE_LEVEL_005C",
    "DIRECTION_005E",      # route-naming detail, not structural

    # --- border-bridge fields, almost entirely blank for MA interior bridges ---
    "OTHER_STATE_CODE_098A", "OTHER_STATE_PCNT_098B", "OTHR_STATE_STRUC_NO_099",

    # --- redundant rating/administrative duplicates of fields you'll keep ---
    "OPR_RATING_METH_063", "INV_RATING_METH_065",   # method codes, not the rating itself
    "SUFFICIENCY_ASTERC", "STATUS_WITH_10YR_RULE",  # redundant w/ STATUS_NO_10YR_RULE
    "PARALLEL_STRUCTURE_101",

    # --- cost/improvement planning fields: mostly NaN, and any *actual*
    #     completed improvement is now better captured by your
    #     had_maintenance / time_since_last_maintenance / maintenance_in_progress
    #     columns rather than these planning-stage NBI fields ---
    "WORK_PROPOSED_075A", "WORK_DONE_BY_075B", "IMP_LEN_MT_076",
    "BRIDGE_IMP_COST_094", "ROADWAY_IMP_COST_095", "TOTAL_IMP_COST_096",
    "YEAR_OF_IMP_097",

    # --- LEAKAGE RISK: appraisal/eval fields that are themselves downstream
    #     outputs of condition assessment, same logic as your Cox model's
    #     "drop leakage-risk appraisal columns" decision ---
    "STRUCTURAL_EVAL_067", "DECK_GEOMETRY_EVAL_068", "UNDCLRENCE_EVAL_069",
    "WATERWAY_EVAL_071", "APPR_ROAD_EVAL_072", "POSTING_EVAL_070",
    "STATUS_NO_10YR_RULE",   # SD/FO status is *derived from* condition ratings
    "BRIDGE_CONDITION", "LOWEST_RATING",  # also derived summary fields (post-58/59/60)
    "SUFFICIENCY_RATING",    # composite score partly built from condition ratings

    # --- inspection program logistics, not deterioration-relevant ---
    "INSPECT_FREQ_MONTHS_091", "FRACTURE_092A", "UNDWATER_LOOK_SEE_092B",
    "SPEC_INSPECT_092C", "FRACTURE_LAST_DATE_093A", "UNDWATER_LAST_DATE_093B",
    "SPEC_LAST_DATE_093C",

    # --- future/forecast fields (not known at observation time -> leakage) ---
    "FUTURE_ADT_114", "YEAR_OF_FUTURE_ADT_115",

    # --- rarely populated navigation fields (mostly N/A unless waterway w/ permit) ---
    "NAV_VERT_CLR_MT_039", "NAV_HORR_CLR_MT_040", "MIN_NAV_CLR_MT_116",
]

existing_drop_cols = [c for c in drop_cols if c in df.columns]
missing_drop_cols = [c for c in drop_cols if c not in df.columns]
if missing_drop_cols:
    print(f"Note: {len(missing_drop_cols)} columns in drop list not found in df (already absent): {missing_drop_cols}")

df = df.drop(columns=existing_drop_cols)
print(f"After dropping {len(existing_drop_cols)} columns: {df.shape}")

# -----------------------------------------------------------------------
# 2. Replace NBI sentinel missing-value codes with NaN
# -----------------------------------------------------------------------
# Per the NBI dictionary, "99" / "99.99" / "999" / "9999" generally mean
# "Miscoded data" (true missing), distinct from "N" which usually means
# "Not applicable" (a structurally meaningful absence, not missingness).
# We'll convert numeric sentinels to NaN but keep "N" as a distinct category
# rather than blindly NaN-ing it everywhere, since "N" carries information
# (e.g., CULVERT_COND_062 = 'N' means "not a culvert", which is meaningful).

numeric_sentinel_cols_99 = [
    "ROUTE_PREFIX_005B", "DIRECTION_005E", "HISTORY_037",
    "DESIGN_LOAD_031", "MEDIAN_CODE_033", "STRUCTURE_FLARED_035",
    "STRUCTURE_KIND_043A", "STRUCTURE_TYPE_043B", "APPR_KIND_044A",
    "APPR_TYPE_044B", "SERVICE_ON_042A", "SERVICE_UND_042B",
    "BASE_HWY_NETWORK_012", "TOLL_020", "FUNCTIONAL_CLASS_026",
    "STRAHNET_HIGHWAY_100", "TRAFFIC_DIRECTION_102", "HIGHWAY_SYSTEM_104",
    "FEDERAL_LANDS_105", "DECK_STRUCTURE_TYPE_107", "NATIONAL_NETWORK_110",
    "PIER_PROTECTION_111",
]
for col in numeric_sentinel_cols_99:
    if col in df.columns:
        df[col] = df[col].replace(99, np.nan).replace("99", np.nan)

# Clearance fields use 99.99 as "no restriction" - this is NOT missing data,
# it's a legitimate "unrestricted" value, so we leave it but flag it via
# a boolean indicator instead of NaN-ing it (NaN would discard real info).
clearance_999_cols = ["MIN_VERT_CLR_010", "VERT_CLR_OVER_MT_053"]
for col in clearance_999_cols:
    if col in df.columns:
        df[f"{col}_unrestricted"] = (df[col] == 99.99).astype(int)
        # Leave the original value as-is; RSF can learn from the 99.99 flag

# Skew angle: 99 = "major variation", legitimately different from missing.
# Flag it, then treat as NaN for the numeric skew value itself (since the
# actual angle isn't known when "major variation" is coded).
if "DEGREES_SKEW_034" in df.columns:
    df["skew_major_variation"] = (df["DEGREES_SKEW_034"] == 99).astype(int)
    df["DEGREES_SKEW_034"] = df["DEGREES_SKEW_034"].replace(99, np.nan)

# Inventory rating: 999 = "live load insignificant" (a legitimate value
# for buried/culvert structures), not missing -> flag separately.
if "INVENTORY_RATING_066" in df.columns:
    df["inventory_rating_insignificant_load"] = (
        df["INVENTORY_RATING_066"] == 999
    ).astype(int)
    df["INVENTORY_RATING_066"] = df["INVENTORY_RATING_066"].replace(999, np.nan)

# -----------------------------------------------------------------------
# 3. Condition ratings (Items 58-62): convert "N" (not applicable, e.g.
#    CULVERT_COND for non-culverts) and "99" (miscoded) to NaN, but
#    keep them as numeric for RSF use. "N" truly means "this component
#    doesn't exist for this bridge" so NaN is the correct encoding here
#    (RSF can split on missingness; that's informative in itself).
# -----------------------------------------------------------------------
condition_cols = [
    "DECK_COND_058", "SUPERSTRUCTURE_COND_059", "SUBSTRUCTURE_COND_060",
    "CHANNEL_COND_061", "CULVERT_COND_062",
]
for col in condition_cols:
    if col in df.columns:
        df[col] = df[col].replace({"N": np.nan, "99": np.nan, 99: np.nan})
        df[col] = pd.to_numeric(df[col], errors="coerce")

# -----------------------------------------------------------------------
# 4. Clean STRUCTURE_NUMBER_008 (strip whitespace - it's read in with
#    leading spaces per the CSV you showed, e.g. "   001111002101")
# -----------------------------------------------------------------------
if "STRUCTURE_NUMBER_008" in df.columns:
    df["STRUCTURE_NUMBER_008"] = df["STRUCTURE_NUMBER_008"].astype(str).str.strip()

# -----------------------------------------------------------------------
# 5. COUNTY_CODE_003 and PLACE_CODE_004: keep as categorical codes (not
#    true numeric quantities), cast to string/category dtype
# -----------------------------------------------------------------------
for col in ["COUNTY_CODE_003", "PLACE_CODE_004", "HIGHWAY_DISTRICT_002"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64").astype(str).replace("<NA>", np.nan)

# -----------------------------------------------------------------------
# 6. YEAR_RECONSTRUCTED_106: 0 means "never reconstructed" per the data
#    dictionary - this is meaningful, not missing. Convert to a flag +
#    keep 0 as-is (or NaN it and rely on the flag, your choice - flag
#    is created either way so RSF can use it cleanly).
# -----------------------------------------------------------------------
if "YEAR_RECONSTRUCTED_106" in df.columns:
    df["ever_reconstructed"] = (df["YEAR_RECONSTRUCTED_106"] > 0).astype(int)
    df["years_since_reconstruction"] = np.where(
        df["YEAR_RECONSTRUCTED_106"] > 0,
        df["SURVEY_YEAR"] - df["YEAR_RECONSTRUCTED_106"],
        np.nan,
    )

# -----------------------------------------------------------------------
# 7. Final dtype pass: ensure key numeric modeling columns are numeric
# -----------------------------------------------------------------------
numeric_cols_to_force = [
    "YEAR_BUILT_027", "ADT_029", "YEAR_ADT_030", "TRAFFIC_LANES_ON_028A",
    "TRAFFIC_LANES_UND_028B", "DEGREES_SKEW_034", "MAIN_UNIT_SPANS_045",
    "APPR_SPANS_046", "MAX_SPAN_LEN_MT_048", "STRUCTURE_LEN_MT_049",
    "DECK_WIDTH_MT_052", "PERCENT_ADT_TRUCK_109", "DECK_AREA",
    "DECK_COND_058", "SUPERSTRUCTURE_COND_059", "SUBSTRUCTURE_COND_060",
]
for col in numeric_cols_to_force:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# -----------------------------------------------------------------------
# 8. Engineer bridge age at time of observation (key survival covariate)
# -----------------------------------------------------------------------
if "YEAR_BUILT_027" in df.columns and "SURVEY_YEAR" in df.columns:
    df["bridge_age"] = df["SURVEY_YEAR"] - df["YEAR_BUILT_027"]
    # Negative ages indicate data errors (survey before construction) - flag, don't silently drop
    df["bridge_age_negative_flag"] = (df["bridge_age"] < 0).astype(int)

# -----------------------------------------------------------------------
# 9. Missingness report
# -----------------------------------------------------------------------
missing_report = (
    df.isna().sum().sort_values(ascending=False) / len(df) * 100
).round(2)
missing_report = missing_report[missing_report > 0]
print("\n--- Missingness report (% of rows) ---")
print(missing_report.to_string())

df.to_csv(OUTPUT_FILE, index=False)
print(f"\nSaved cleaned dataset: {OUTPUT_FILE}")
print(f"Final shape: {df.shape}")

In [ ]:
cols = [
    'DECK_COND_058',
    'SUPERSTRUCTURE_COND_059',
    'SUBSTRUCTURE_COND_060'
]

# Replace N with 10
df[cols] = df[cols].replace('N', 10)

# Convert to numeric
df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')

# Verify conversion worked
print(df[cols].dtypes)